# Netflix Titles — Data Collection & Cleaning (Phases 1-2)

This notebook documents and demonstrates the cleaning pipeline that lives
in `src/data_cleaning.py` (the single source of truth used by every other
notebook, the SQL loader, and the dashboard). Running this notebook shows
you exactly what changes between raw and clean data, with before/after
evidence at every step.

In [1]:
import sys
sys.path.append("..")
import pandas as pd
from src.data_cleaning import (
    load_raw, standardize_columns, clean_text_fields, handle_missing_values,
    remove_duplicates, standardize_dates, parse_duration, standardize_categories,
    build_exploded_tables, RAW_PATH
)

pd.set_option("display.max_columns", 30)

## Phase 1 — Data Collection
Source: Netflix Titles dataset (Shivam Bansal / Kaggle, mirrored via the
`rfordatascience/tidytuesday` GitHub repository for reproducible access).
7,787 raw rows, 12 source columns.

In [2]:
raw = load_raw(RAW_PATH)
print("Shape:", raw.shape)
raw.head(3)

Shape: (7787, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,"August 14, 2020",2020,TV-MA,4 Seasons,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s2,Movie,7:19,Jorge Michel Grau,"Demián Bichir, Héctor Bonilla, Oscar Serrano, ...",Mexico,"December 23, 2016",2016,TV-MA,93 min,"Dramas, International Movies",After a devastating earthquake hits Mexico Cit...
2,s3,Movie,23:59,Gilbert Chan,"Tedd Chan, Stella Chung, Henley Hii, Lawrence ...",Singapore,"December 20, 2018",2011,R,78 min,"Horror Movies, International Movies","When an army recruit is found dead, his fellow..."


In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 7787 entries, 0 to 7786
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       7787 non-null   str  
 1   type          7787 non-null   str  
 2   title         7787 non-null   str  
 3   director      5398 non-null   str  
 4   cast          7069 non-null   str  
 5   country       7280 non-null   str  
 6   date_added    7777 non-null   str  
 7   release_year  7787 non-null   int64
 8   rating        7780 non-null   str  
 9   duration      7787 non-null   str  
 10  listed_in     7787 non-null   str  
 11  description   7787 non-null   str  
dtypes: int64(1), str(11)
memory usage: 3.4 MB


## Phase 2 — Data Cleaning
### Step 1: Handle missing values

In [4]:
print("Missing values BEFORE cleaning:")
print(raw.isnull().sum()[raw.isnull().sum() > 0])

df = standardize_columns(raw)
df = clean_text_fields(df)
df = handle_missing_values(df)

print("\nMissing values AFTER handle_missing_values():")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nStrategy: director/cast/country -> 'Unknown' placeholder (genuinely unrecorded, "
      "not numerically interpolable). rating -> 'Not Rated'. date_added left as NaT and "
      "flagged via date_added_missing, since the original release date is NOT a safe "
      "substitute for 'when it was added to Netflix' in trend analysis.")

Missing values BEFORE cleaning:
director      2389
cast           718
country        507
date_added      10
rating           7
dtype: int64

Missing values AFTER handle_missing_values():
date_added    10
dtype: int64

Strategy: director/cast/country -> 'Unknown' placeholder (genuinely unrecorded, not numerically interpolable). rating -> 'Not Rated'. date_added left as NaT and flagged via date_added_missing, since the original release date is NOT a safe substitute for 'when it was added to Netflix' in trend analysis.


### Step 2: Remove duplicates

In [5]:
before = len(df)
df = remove_duplicates(df)
print(f"Rows before dedup: {before}  ->  after: {len(df)}  "
      f"(removed {df.attrs.get('duplicates_removed', 0)} exact/fuzzy duplicates)")

Rows before dedup: 7787  ->  after: 7786  (removed 1 exact/fuzzy duplicates)


### Step 3: Standardize date formats
Source dates arrive as free text like `"August 14, 2020"`. We parse to a
real datetime and derive year/month/quarter/day-of-week helper columns so
every downstream notebook uses identical date logic.

In [6]:
print("Sample raw date_added values:", df["date_added"].dropna().unique()[:3].tolist())
df = standardize_dates(df)
df[["date_added", "date_added_parsed", "year_added", "month_name_added", "day_of_week_added"]].dropna().head(3)

Sample raw date_added values: ['August 14, 2020', 'December 23, 2016', 'December 20, 2018']


,date_added,date_added_parsed,year_added,month_name_added,day_of_week_added
0,"August 14, 2020",2020-08-14,2020.0,August,Friday
1,"December 23, 2016",2016-12-23,2016.0,December,Friday
2,"December 20, 2018",2018-12-20,2018.0,December,Thursday


### Step 4: Parse duration into comparable numeric fields
`duration` mixes two incompatible units — "90 min" for movies and
"3 Seasons" for TV shows. We split into two explicit columns rather than
coercing into one misleading number.

In [7]:
print("Raw duration samples:", df["duration"].unique()[:6].tolist())
df = parse_duration(df)
df[["type", "duration", "duration_minutes", "duration_seasons"]].drop_duplicates(subset=["type"]).head(2)

Raw duration samples: ['4 Seasons', '93 min', '78 min', '80 min', '123 min', '1 Season']


,type,duration,duration_minutes,duration_seasons
0,TV Show,4 Seasons,NaN,4.0
1,Movie,93 min,93.0,NaN


### Step 5: Detect & standardize inconsistent categories
The `rating` column mixes movie ratings (PG-13, R) with TV ratings
(TV-MA, TV-14) and legacy synonyms (`UR` vs `NR` both mean "Unrated").

In [8]:
print("Raw rating values:", sorted(df["rating"].dropna().unique().tolist()))
df = standardize_categories(df)
print("\nDerived maturity_level buckets:", df["maturity_level"].value_counts().to_dict())
print("\nNote: 'UR' was merged into 'NR' (both mean Unrated) — see RATING_FIXES in src/data_cleaning.py.")

Raw rating values: ['G', 'NC-17', 'NR', 'Not Rated', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'UR']

Derived maturity_level buckets: {'Adults': 3530, 'Teens': 2317, 'Family': 1053, 'Kids': 790, 'Unrated': 96}

Note: 'UR' was merged into 'NR' (both mean Unrated) — see RATING_FIXES in src/data_cleaning.py.


### Step 6: Build exploded bridge tables
`country`, `cast`, `director`, and `listed_in` are comma-separated
multi-valued fields. We explode them into long-format bridge tables so
SQL JOINs and Pandas groupbys attribute a title to ALL of its countries/
genres/cast members, not just the first one.

In [9]:
exploded = build_exploded_tables(df)
for name, tbl in exploded.items():
    print(f"bridge_{name}: {len(tbl)} rows (vs {df.shape[0]} titles — confirms multi-value expansion)")
exploded["country"].head(5)

bridge_country: 9066 rows (vs 7786 titles — confirms multi-value expansion)
bridge_genre: 17068 rows (vs 7786 titles — confirms multi-value expansion)
bridge_cast: 55947 rows (vs 7786 titles — confirms multi-value expansion)
bridge_director: 6114 rows (vs 7786 titles — confirms multi-value expansion)


,show_id,country_name
0,s1,Brazil
1,s2,Mexico
2,s3,Singapore
3,s4,United States
4,s5,United States


## Final Cleaned Dataset

In [10]:
print("Final shape:", df.shape)
df.isnull().sum()[df.isnull().sum() > 0]

Final shape: (7786, 25)


date_added             10
date_added_parsed      10
year_added             10
month_added            10
month_name_added       10
quarter_added          10
day_of_week_added      10
duration_minutes     2410
duration_seasons     5376
dtype: int64

Remaining "missing" values above are all expected and correct:
`duration_minutes` is null for TV Shows (they use `duration_seasons`
instead) and vice versa, and `date_added*` fields are null for the 10
titles where Netflix never recorded an add-date — these are excluded from
time-trend analysis, not silently imputed.

Run `python src/data_cleaning.py` from the project root to regenerate
`data/netflix_titles_clean.csv` and the four `data/bridge_*.csv` files
used everywhere else in this project.